# 02 — Embedding Extraction, Validation Factors, Uncertainty Selection & Lightweight MiewID Adaptation
Extracts embeddings for `MiewID-msv3` and `MegaDescriptor-L-384`, evaluates H1/H2/H3/H4 strictly on **Validation** (never touching locked test days `2023-08-19`/`20` or `NyalaData` test), locks the winning confidence rule, and trains a lightweight residual projection head strictly on Gallery days `14–16`.

In [ ]:
import json, os, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sklearn.metrics import roc_auc_score, roc_curve

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class MiewIDBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("efficientnetv2_rw_m", pretrained=False, num_classes=0)
        self.backbone.global_pool = GeM()
        self.bn = nn.BatchNorm1d(2152)
    def forward(self, x):
        return self.bn(self.backbone(x).view(x.shape[0], -1))

def load_miewid_msv3(device):
    sf_path = hf_hub_download("conservationxlabs/miewid-msv3", "model.safetensors")
    sd = load_file(sf_path)
    feat_sd = {k: v for k, v in sd.items() if k.startswith("backbone.") or k.startswith("bn.")}
    m = MiewIDBackbone()
    missing, unexpected = m.load_state_dict(feat_sd, strict=True)
    assert len(missing) == 0 and len(unexpected) == 0, f"Mismatch: {missing} | {unexpected}"
    print(f"Loaded official conservationxlabs/miewid-msv3 ({len(feat_sd)} tensors, GeM p={m.backbone.global_pool.p.item():.3f})")
    return m.to(device).eval()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

OUT = Path("/kaggle/working/out")
OUT.mkdir(parents=True, exist_ok=True)

# Locate splits from step 01
split_cows_path = next(Path("/kaggle/input").rglob("splits_cows.csv"))
split_nyala_path = next(Path("/kaggle/input").rglob("splits_nyala.csv"))
cows = pd.read_csv(split_cows_path)
nyala = pd.read_csv(split_nyala_path)
print("Loaded splits:", len(cows), "cows,", len(nyala), "nyala")

# Keep ONLY gallery + val rows for step 02 (STRICT ZERO TEST TOUCH!)
cows_gv = cows[cows["role"].isin(["gallery", "val_known", "val_unknown"]) | (cows["random_split_role"].isin(["rand_gallery", "rand_query"]))].copy().reset_index(drop=True)
nyala_tr = nyala[nyala["split"] == "train"].copy().reset_index(drop=True)
print("Cows gallery+val subset:", len(cows_gv), "| Nyala train subset:", len(nyala_tr))

def apply_degradation(img, mode):
    if mode == "clean":
        return img
    if mode == "low_light":
        return ImageEnhance.Brightness(img).enhance(0.35)
    if mode == "blur":
        return img.filter(ImageFilter.GaussianBlur(radius=3.0))
    if mode == "crop70":
        w, h = img.size
        nw, nh = int(w * 0.70), int(h * 0.70)
        left, top = (w - nw) // 2, (h - nh) // 2
        return img.crop((left, top, left + nw, top + nh))
    raise ValueError(mode)

class ImgDS(Dataset):
    def __init__(self, paths, tfm, mode="clean"):
        self.paths = list(paths)
        self.tfm = tfm
        self.mode = mode
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            im = im.convert("RGB")
            if self.mode != "clean":
                im = apply_degradation(im, self.mode)
            return self.tfm(im)

@torch.no_grad()
def extract_embeddings(model, paths, tfm, batch_size=64, mode="clean"):
    ds = ImgDS(paths, tfm, mode=mode)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    embs = []
    for xb in dl:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            out = model(xb)
        if isinstance(out, (tuple, list)):
            out = out[0]
        elif hasattr(out, "last_hidden_state"):
            out = out.last_hidden_state.mean(dim=1)
        out = F.normalize(out.float(), p=2, dim=-1)
        embs.append(out.cpu().numpy())
    return np.concatenate(embs, axis=0)

# Transforms
miew_tfm = T.Compose([
    T.Resize((440, 440)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
mega_tfm = T.Compose([
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# 1. Extract MiewID-msv3
t0 = time.time()
print("Loading conservationxlabs/miewid-msv3 ...")
miew = load_miewid_msv3(DEVICE)
miew_cows_gv = extract_embeddings(miew, cows_gv["abs_path"], miew_tfm, batch_size=48)
miew_nyala_tr = extract_embeddings(miew, nyala_tr["abs_path"], miew_tfm, batch_size=48)
print(f"MiewID extracted in {time.time()-t0:.1f}s | dim={miew_cows_gv.shape[1]}")

# Also extract degraded val_known queries for MiewID
val_known_mask = (cows_gv["role"] == "val_known").to_numpy()
val_known_paths = cows_gv.loc[val_known_mask, "abs_path"].tolist()
miew_deg = {}
for mode in ["low_light", "blur", "crop70"]:
    miew_deg[mode] = extract_embeddings(miew, val_known_paths, miew_tfm, batch_size=48, mode=mode)
del miew; torch.cuda.empty_cache()

# 2. Extract MegaDescriptor-L-384
t0 = time.time()
print("Loading BVRA/MegaDescriptor-L-384 ...")
mega = timm.create_model("hf-hub:BVRA/MegaDescriptor-L-384", pretrained=True).to(DEVICE).eval()
mega_cows_gv = extract_embeddings(mega, cows_gv["abs_path"], mega_tfm, batch_size=48)
mega_nyala_tr = extract_embeddings(mega, nyala_tr["abs_path"], mega_tfm, batch_size=48)
print(f"MegaDescriptor extracted in {time.time()-t0:.1f}s | dim={mega_cows_gv.shape[1]}")

mega_deg = {}
for mode in ["low_light", "blur", "crop70"]:
    mega_deg[mode] = extract_embeddings(mega, val_known_paths, mega_tfm, batch_size=48, mode=mode)
del mega; torch.cuda.empty_cache()

# Evaluation utilities
def eval_retrieval(q_emb, q_ids, g_emb, g_ids, q_cams=None, g_cams=None, cam_mode="all"):
    sims = q_emb @ g_emb.T  # (Nq, Ng)
    n_q = len(q_ids)
    r1 = np.zeros(n_q, dtype=float)
    r5 = np.zeros(n_q, dtype=float)
    aps = np.zeros(n_q, dtype=float)
    top1_sim = np.zeros(n_q, dtype=float)
    id_margin = np.zeros(n_q, dtype=float)
    valid = np.ones(n_q, dtype=bool)

    g_ids_arr = np.asarray(g_ids)
    for i in range(n_q):
        row = sims[i].copy()
        if cam_mode == "same" and q_cams is not None:
            mask = (g_cams == q_cams[i])
        elif cam_mode == "cross" and q_cams is not None:
            mask = (g_cams != q_cams[i])
        else:
            mask = np.ones(len(g_ids_arr), dtype=bool)
        if not np.any(mask & (g_ids_arr == q_ids[i])):
            valid[i] = False
            continue
        row[~mask] = -1e9
        order = np.argsort(-row)
        ranked_ids = g_ids_arr[order]
        ranked_sims = row[order]

        r1[i] = float(ranked_ids[0] == q_ids[i])
        r5[i] = float(q_ids[i] in ranked_ids[:5])
        matches = (ranked_ids == q_ids[i])
        rel_idx = np.where(matches)[0]
        prec_at_k = (np.arange(1, len(rel_idx) + 1)) / (rel_idx + 1)
        aps[i] = float(prec_at_k.mean())

        top1_sim[i] = float(ranked_sims[0])
        top1_id = ranked_ids[0]
        # second-best DISTINCT identity similarity
        other_mask = (ranked_ids != top1_id) & (ranked_sims > -1e8)
        second_sim = float(ranked_sims[other_mask][0]) if np.any(other_mask) else 0.0
        id_margin[i] = top1_sim[i] - second_sim

    return {
        "n_valid": int(valid.sum()),
        "r1": float(r1[valid].mean()),
        "r5": float(r5[valid].mean()),
        "mAP": float(aps[valid].mean()),
        "r1_arr": r1[valid],
        "top1_sim": top1_sim[valid],
        "id_margin": id_margin[valid],
    }

def bootstrap_ci(arr, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(arr), size=(n_boot, len(arr)))
    means = arr[idx].mean(axis=1)
    return [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)]

def openset_scores(known_emb, unk_emb, g_emb, g_ids):
    g_ids_arr = np.asarray(g_ids)
    def score_batch(emb):
        sims = emb @ g_emb.T
        t1, mg = [], []
        for i in range(len(emb)):
            order = np.argsort(-sims[i])
            r_ids = g_ids_arr[order]
            r_sims = sims[i][order]
            t1.append(float(r_sims[0]))
            other = r_sims[r_ids != r_ids[0]]
            mg.append(float(r_sims[0] - (other[0] if len(other) else 0.0)))
        return np.array(t1), np.array(mg)
    kt1, kmg = score_batch(known_emb)
    ut1, umg = score_batch(unk_emb)
    y = np.concatenate([np.ones(len(kt1)), np.zeros(len(ut1))])  # 1 = known, 0 = unknown
    res = {}
    for name, s in [("top1_sim", np.concatenate([kt1, ut1])), ("id_margin", np.concatenate([kmg, umg]))]:
        auc = float(roc_auc_score(y, s))
        # Unknown detection TPR at 5% FPR (invert: higher score = unknown)
        fpr, tpr, _ = roc_curve(1 - y, -s)
        tpr_at_5 = float(tpr[np.searchsorted(fpr, 0.05, side="right") - 1]) if np.any(fpr <= 0.05) else 0.0
        res[name] = {"auroc": round(auc, 4), "unk_tpr_at_5fpr": round(tpr_at_5, 4)}
    return res

# 3. Evaluate zero-shot MiewID & MegaDescriptor on Validation
gal_m = (cows_gv["role"] == "gallery").to_numpy()
vk_m = (cows_gv["role"] == "val_known").to_numpy()
vu_m = (cows_gv["role"] == "val_unknown").to_numpy()
rg_m = (cows_gv["random_split_role"] == "rand_gallery").to_numpy()
rq_m = (cows_gv["random_split_role"] == "rand_query").to_numpy()

g_ids = cows_gv.loc[gal_m, "identity"].to_numpy()
g_cams = cows_gv.loc[gal_m, "camera"].to_numpy()
vk_ids = cows_gv.loc[vk_m, "identity"].to_numpy()
vk_cams = cows_gv.loc[vk_m, "camera"].to_numpy()

val_report = {}
for mname, emb, deg_dict in [("miewid_msv3", miew_cows_gv, miew_deg), ("megadescriptor_l384", mega_cows_gv, mega_deg)]:
    res_rand = eval_retrieval(emb[rq_m], cows_gv.loc[rq_m, "identity"].to_numpy(), emb[rg_m], cows_gv.loc[rg_m, "identity"].to_numpy())
    res_day = eval_retrieval(emb[vk_m], vk_ids, emb[gal_m], g_ids, vk_cams, g_cams, "all")
    res_same = eval_retrieval(emb[vk_m], vk_ids, emb[gal_m], g_ids, vk_cams, g_cams, "same")
    res_cross = eval_retrieval(emb[vk_m], vk_ids, emb[gal_m], g_ids, vk_cams, g_cams, "cross")

    # Closed-set error flagging AUROC (1 = error, 0 = correct -> higher negative confidence = error)
    y_err = 1 - res_day["r1_arr"]
    err_auc_t1 = round(float(roc_auc_score(y_err, -res_day["top1_sim"])), 4) if 0 < y_err.sum() < len(y_err) else None
    err_auc_mg = round(float(roc_auc_score(y_err, -res_day["id_margin"])), 4) if 0 < y_err.sum() < len(y_err) else None

    os_res = openset_scores(emb[vk_m], emb[vu_m], emb[gal_m], g_ids)

    deg_res = {}
    for dmode, demb in deg_dict.items():
        dr = eval_retrieval(demb, vk_ids, emb[gal_m], g_ids)
        deg_res[dmode] = {"r1": round(dr["r1"], 4), "r5": round(dr["r5"], 4), "mAP": round(dr["mAP"], 4), "drop_r1": round(res_day["r1"] - dr["r1"], 4)}

    val_report[mname] = {
        "random_split": {"r1": round(res_rand["r1"], 4), "r1_ci": bootstrap_ci(res_rand["r1_arr"]), "mAP": round(res_rand["mAP"], 4)},
        "cross_day_val": {"r1": round(res_day["r1"], 4), "r1_ci": bootstrap_ci(res_day["r1_arr"]), "r5": round(res_day["r5"], 4), "mAP": round(res_day["mAP"], 4)},
        "same_cam_val": {"r1": round(res_same["r1"], 4), "r1_ci": bootstrap_ci(res_same["r1_arr"]), "mAP": round(res_same["mAP"], 4)},
        "cross_cam_val": {"r1": round(res_cross["r1"], 4), "r1_ci": bootstrap_ci(res_cross["r1_arr"]), "mAP": round(res_cross["mAP"], 4)},
        "day_drop_r1": round(res_rand["r1"] - res_day["r1"], 4),
        "cam_drop_r1": round(res_same["r1"] - res_cross["r1"], 4),
        "degradations_val": deg_res,
        "closed_set_error_auroc": {"top1_sim": err_auc_t1, "id_margin": err_auc_mg},
        "openset_unknown_val": os_res,
    }

# 4. Train lightweight residual cosine head on frozen MiewID embeddings STRICTLY on Gallery days 14-16!
class ResidualAdapter(nn.Module):
    def __init__(self, dim=2152, hidden=512, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.alpha = nn.Parameter(torch.tensor(0.25))
    def forward(self, x):
        return F.normalize(x + self.alpha * self.net(x), p=2, dim=-1)

X_tr = torch.tensor(miew_cows_gv[gal_m], dtype=torch.float32, device=DEVICE)
uniq_ids = sorted(set(g_ids))
id2y = {k: i for i, k in enumerate(uniq_ids)}
y_tr = torch.tensor([id2y[k] for k in g_ids], dtype=torch.long, device=DEVICE)
dim = X_tr.shape[1]

torch.manual_seed(SEED)
adapter = ResidualAdapter(dim=dim, hidden=512, dropout=0.25).to(DEVICE)
# Cosine classifier weights (ArcFace-style angular margin)
W_cls = nn.Parameter(F.normalize(torch.randn(len(uniq_ids), dim, device=DEVICE), dim=-1))
opt = torch.optim.AdamW(list(adapter.parameters()) + [W_cls], lr=1e-3, weight_decay=1e-2)
scale, margin = 16.0, 0.25

for epoch in range(120):
    adapter.train()
    perm = torch.randperm(len(X_tr), device=DEVICE)
    for b in range(0, len(X_tr), 128):
        idx = perm[b:b+128]
        z = adapter(X_tr[idx])
        w = F.normalize(W_cls, dim=-1)
        cos = torch.clamp(z @ w.T, -1.0 + 1e-6, 1.0 - 1e-6)
        # Additive cosine margin (CosFace) for stability
        onehot = F.one_hot(y_tr[idx], num_classes=len(uniq_ids)).float()
        logits = scale * (cos - onehot * margin)
        loss = F.cross_entropy(logits, y_tr[idx])
        opt.zero_grad(); loss.backward(); opt.step()

adapter.eval()
with torch.no_grad():
    miew_adapt_gv = adapter(torch.tensor(miew_cows_gv, dtype=torch.float32, device=DEVICE)).cpu().numpy()

res_ad_day = eval_retrieval(miew_adapt_gv[vk_m], vk_ids, miew_adapt_gv[gal_m], g_ids, vk_cams, g_cams, "all")
res_ad_same = eval_retrieval(miew_adapt_gv[vk_m], vk_ids, miew_adapt_gv[gal_m], g_ids, vk_cams, g_cams, "same")
res_ad_cross = eval_retrieval(miew_adapt_gv[vk_m], vk_ids, miew_adapt_gv[gal_m], g_ids, vk_cams, g_cams, "cross")
os_ad = openset_scores(miew_adapt_gv[vk_m], miew_adapt_gv[vu_m], miew_adapt_gv[gal_m], g_ids)
y_err_ad = 1 - res_ad_day["r1_arr"]

val_report["miewid_adapted_val"] = {
    "cross_day_val": {"r1": round(res_ad_day["r1"], 4), "r1_ci": bootstrap_ci(res_ad_day["r1_arr"]), "r5": round(res_ad_day["r5"], 4), "mAP": round(res_ad_day["mAP"], 4)},
    "same_cam_val": {"r1": round(res_ad_same["r1"], 4), "mAP": round(res_ad_same["mAP"], 4)},
    "cross_cam_val": {"r1": round(res_ad_cross["r1"], 4), "r1_ci": bootstrap_ci(res_ad_cross["r1_arr"]), "mAP": round(res_ad_cross["mAP"], 4)},
    "gain_cross_cam_r1": round(res_ad_cross["r1"] - val_report["miewid_msv3"]["cross_cam_val"]["r1"], 4),
    "closed_set_error_auroc": {
        "top1_sim": round(float(roc_auc_score(y_err_ad, -res_ad_day["top1_sim"])), 4) if 0 < y_err_ad.sum() < len(y_err_ad) else None,
        "id_margin": round(float(roc_auc_score(y_err_ad, -res_ad_day["id_margin"])), 4) if 0 < y_err_ad.sum() < len(y_err_ad) else None,
    },
    "openset_unknown_val": os_ad,
}

# Lock winning confidence score based on MiewID validation AUROC
m_err = val_report["miewid_msv3"]["closed_set_error_auroc"]
m_os = val_report["miewid_msv3"]["openset_unknown_val"]
locked_rules = {
    "locked_closed_set_score": "id_margin" if (m_err["id_margin"] or 0) >= (m_err["top1_sim"] or 0) else "top1_sim",
    "locked_openset_score": "top1_sim" if m_os["top1_sim"]["auroc"] >= m_os["id_margin"]["auroc"] else "id_margin",
    "adapter_dim": dim,
}
val_report["locked_rules_for_test"] = locked_rules

# Save adapter weights, gallery embeddings, and validation summary
torch.save({"state_dict": adapter.state_dict(), "dim": dim}, OUT / "miewid_adapter.pt")
np.savez_compressed(
    OUT / "gallery_embeddings.npz",
    miew_gal=miew_cows_gv[gal_m],
    mega_gal=mega_cows_gv[gal_m],
    miew_adapt_gal=miew_adapt_gv[gal_m],
    g_ids=g_ids,
    g_cams=g_cams,
)
(OUT / "val_summary.json").write_text(json.dumps(val_report, indent=2))
print(json.dumps(val_report, indent=2))